# 05 — Two-objective Pareto optimisation

This notebook adds an **optimisation / search layer** on top of the existing agent-based simulation. The ABM itself is not modified — it is reused as the evaluation function `x → (C(x), L(x))`.

## Notation (full statement in `src/optimisation.py` docstring)

- $G = (V(G), E(G))$ — the directed road graph (largest strongly connected component of OSMnx).
- $V(G)$ — node set (drivable junctions / vertices).
- $x \subset V(G) \times \mathbb{N}^+$ — a **candidate layout** = a finite set of `(node, ports)` pairs.
- $K(x) = |x|$ — number of stations.
- $P(x) = \sum_{(n, p) \in x} p$ — total number of ports.

## Two-objective problem

$$\text{minimise } C(x) = c_{\text{station}}\,K(x) + c_{\text{port}}\,P(x)$$
$$\text{minimise } L(x) = w_1\,\hat{L}_{\text{wait}}(x) + w_2\,\hat{L}_{\text{detour}}(x) + w_3\,\hat{L}_{95}(x)$$

where each $\hat{L}_\cdot$ is the corresponding raw user-burden metric from the ABM summary, min-max normalised across the candidate set. The four-component objective used earlier in the thesis (W, D, Q, U_imb) is **collapsed to two dimensions** for the Pareto analysis (cost vs. service loss). The four raw components are still reported as diagnostics.

## Scalarisation and α-sweep

$$J_\alpha(x) = \alpha\,\hat{C}(x) + (1 - \alpha)\,\hat{L}(x), \quad \alpha \in [0, 1]$$

$\alpha \to 1$ prioritises cost; $\alpha \to 0$ prioritises service quality.

## Search method

**Structured candidate search**, not a genetic algorithm. We enumerate the $(K, P, \text{pattern})$ grid:
- $K \in \{8, 12, 16, 20, 24\}$,
- $P / K \in \{1.0, 1.7, 2.5\}$,
- pattern $\in \{\text{random}, \text{clustered}, \text{distributed}\}$,

plus the real OCM layout (`real_S1`). All candidates are evaluated with the same agent population (`OPT.eval_n_agents`), the same RNG seed, and the same simulated horizon, so differences in $C, L$ are attributable to the layout alone.

Why not a GA? The structured search already spans the cost/quality grid coarsely, and a thesis-grade GA implementation would require population-size / crossover-scheme / stopping-rule decisions that are out of scope for the preliminary results in this chapter. A GA could be plugged into the same evaluation interface (`evaluate_layout`) without changing the Pareto or α-sweep machinery.

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

import pandas as pd
from src import config, graph_utils, charger_data, optimisation as opt

print('OptimisationParams defaults:')
for k, v in opt.OPT.__dict__.items():
    print(f'  {k:24s} = {v}')

In [ ]:
G = graph_utils.get_or_build_graph()
df_clean = charger_data.load_clean_chargers()
print(f'graph: {G.number_of_nodes():,} nodes  |  clean chargers: {len(df_clean)}  |  total ports: {int(df_clean["number_of_points"].sum())}')

## Run the experiment

Generates the candidate set, runs the ABM on each candidate (with a reduced agent budget for runtime), and computes the Pareto frontier and α-sweep. Total runtime ≈ `len(candidates) × per-candidate ABM time`; tune `opt.OPT.eval_n_agents` if needed.

In [ ]:
df_candidates, df_alpha = opt.run_optimisation_experiment(G, df_clean, verbose=True)
print(f'{len(df_candidates)} candidates evaluated; {int(df_candidates["pareto"].sum())} are Pareto-efficient.')

In [ ]:
df_candidates.sort_values(['pareto', 'C'], ascending=[False, True]).head(20)

## α-sweep — best layout per α

In [ ]:
df_alpha

## Save tables and figures

In [ ]:
paths = opt.write_all_optimisation_outputs(df_candidates, df_alpha)
for k, v in paths.items():
    print(f'{k:36s} → {v}')

In [ ]:
from IPython.display import Image, display
for name in ['pareto_frontier_cost_quality.png', 'alpha_tradeoff_solutions.png']:
    p = config.FIGURES_DIR / name
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f'missing: {p}')

## Interpretation

- The **Pareto frontier** shows the cost / service-loss layouts that are not dominated: every layout on the frontier achieves strictly lower service loss only at the cost of higher infrastructure spend, and vice versa.
- The real OCM layout (`real_S1`) is plotted with a blue star. Its position on the scatter shows whether the existing deployment sits on or near the empirically found frontier, or whether reallocating the same total port budget into a different spatial pattern would dominate it on both axes.
- The **α-sweep** identifies the J_α-optimal layout for each weighting of cost vs. service loss. The bar chart on the right of `alpha_tradeoff_solutions.png` summarises how often each candidate is selected across the α grid; layouts that are picked under many α values are the most robust compromise solutions.
- The four-component breakdown from the earlier analysis (W, D, Q, U_imb) is still available in `outputs/tables/scenario_summary.csv`; the two-objective view collapses them into cost and aggregated quality for a clean trade-off figure.